# Calculating ground state properties with **`exciting`** using _GGA, metaGGA_ and _hybrid_ functionals
By: Hannah Kleine, Sebastian Tillack, Kshitij Sinha

**<span style="color:firebrick">Note</span>**: Due to time constraints, all results used in this tutorial have been precomputed. Therefore, the corresponding data are retrieved from the [**NOMAD**](https://nomad-lab.eu/prod/v1/gui/search/entries) database. Otherwise, `exciting` would need to be installed and compiled, and the appropriate environment would need to be configured. More information about exciting and its usage can be found on [exciting webpage](https://www.exciting-code.org/).

<hr style="border:2px solid #DDD"> </hr>

## Introduction and physical background

**Density Functional Theory (DFT)** describes interacting many-electron systems by mapping them onto an auxiliary non-interacting system that reproduces the **same electron density** as the original system. This mapping reduces the complicated many-body problem to an effective single-particle problem described by the **Kohn–Sham (KS) equations**

\begin{equation} \label{eq:kohn_sham}
\bigg[ -\frac{1}{2} \nabla^{2} + v_{\text{KS}}(\boldsymbol{r})\bigg]\psi_{i\boldsymbol{k}}(\boldsymbol{r})
=
\epsilon_{i\boldsymbol{k}}\,\psi_{i\boldsymbol{k}}(\boldsymbol{r}),
\end{equation}

where $\boldsymbol{r}$ denotes the spatial coordinate and $\boldsymbol{k}$ the wave vector.  
$\psi_{i\boldsymbol{k}}(\boldsymbol{r})$ are the Kohn–Sham orbitals, $\epsilon_{i\boldsymbol{k}}$ their corresponding eigenvalues, and $v_{\text{KS}}(\boldsymbol{r})$ is the effective Kohn–Sham potential.

The effective potential consists of three contributions

\begin{equation}
v_{\text{KS}}(\boldsymbol{r}) =
v_{\text{ext}}(\boldsymbol{r}) +
v_{\text{H}}[n](\boldsymbol{r}) +
v_{\text{xc}}[n](\boldsymbol{r}),
\end{equation}

where

- $v_{\text{ext}}$ is the **external potential** generated by the atomic nuclei,
- $v_{\text{H}}$ is the **Hartree potential**, describing the classical electron–electron Coulomb interaction,
- $v_{\text{xc}}$ is the **exchange–correlation (xc) potential**, which contains all many-body effects beyond the Hartree term.

Both $v_{\text{H}}$ and $v_{\text{xc}}$ depend on the **electron density** $n(\boldsymbol{r})$. The electron density itself can be written in terms of the Kohn–Sham orbitals as

\begin{equation}
n(\boldsymbol{r}) =
\sum_{\boldsymbol{k}} w_{\boldsymbol{k}}
\sum_i f_{i\boldsymbol{k}}
|\psi_{i\boldsymbol{k}}(\boldsymbol{r})|^2,
\end{equation}

where $w_{\boldsymbol{k}}$ denotes the weight of the $\boldsymbol{k}$-point and $f_{i\boldsymbol{k}}$ the occupation number of the corresponding state.

In principle, the Kohn–Sham formalism is **exact** and yields the exact ground-state density. In practice, however, the exact form of the exchange–correlation potential $v_{\text{xc}}$ is unknown and must be approximated.

Over the years, several approximations for the exchange–correlation functional have been developed. The simplest one is the **Local Density Approximation (LDA)**, where the xc potential at each point in space is approximated by that of a homogeneous electron gas with the same local electron density. LDA often provides reasonable total energies for metals, but tends to underestimate lattice constants and bond lengths.

An improvement over LDA is the **Generalized Gradient Approximation (GGA)**, which includes not only the local density but also its gradient. This generally leads to improved total energies and more accurate lattice parameters, although band gaps are still typically underestimated.

A further development are **meta-GGA functionals**, which additionally depend on the kinetic-energy density of the Kohn–Sham orbitals. This additional information allows the functional to better distinguish between different bonding environments and often improves structural properties and energetics while keeping the computational cost similar to GGA.

Another way to improve the description of electronic properties is the use of **hybrid functionals**, which mix a fraction of exact exchange from Hartree–Fock theory with the DFT exchange functional. Hybrid functionals often provide significantly improved band gaps, but they are computationally more expensive.

The choice of the appropriate exchange–correlation functional therefore depends on the material system as well as the property of interest. In practical calculations, this choice determines both the accuracy of the results and the computational cost. In this tutorial, we demonstrate calculations using **GGA, meta-GGA, and hybrid functionals**.

Once the crystal structure and the exchange–correlation functional are specified, the ground-state properties of the system can be obtained by solving the Kohn–Sham equations self-consistently. Since the Kohn–Sham potential depends on the electron density, and the electron density itself depends on the Kohn–Sham orbitals, the equations must be solved iteratively until consistency between density and potential is reached.

---

#### DFT Workflow

The starting point of a ground-state calculation is the **crystal structure** of the system. At the beginning of the calculation, an initial electron density is generated from a superposition of atomic densities. Since this initial density neglects the interaction between atoms, it is typically only a rough approximation of the true electron density.

The calculation then proceeds iteratively through the following steps:

1. **Determine the potential**  
   Construct the Kohn–Sham potential from the current electron density.

2. **Solve the Kohn–Sham equations**  
   Solve the Kohn-Sham equations to obtain the eigenvalues, eigenfunctions, and the total energy.

3. **Update the electron density**  
   Calculate a new electron density from the Kohn–Sham orbitals.

4. **Density mixing**  
   Generate an improved density by mixing the current density with densities from previous iterations in order to stabilize the convergence.

5. **Iteration**  
   Repeat the procedure starting again from step (1).

This process continues until the electron density, potential, and total energy no longer change within a predefined threshold. At this point the calculation has reached **self-consistency**, and the procedure is therefore called a **self-consistent field (SCF) calculation**. Each iteration in this process is referred to as an **SCF cycle**.

## Setup and usage of `exciting`

In this tutorial, we demonstrate how to run an **exciting** ground-state calculation on the example of **$\beta$ - Ga<sub>2</sub>O<sub>3</sub>**.
In order to run an exciting calculation, we first need to generate an `input.xml` file which contains all the necessary input parameters. For this, we use the exciting python interface **excitingtools**. Details on this tool can be found in the [**excitingtools tutorial**](https://www.exciting-code.org/uploads/exciting/tutorial_notebooks/tutorial_excitingtools.html).

In [ ]:
from excitingtools.input import (ExcitingStructure, 
                                 ExcitingInputXML, 
                                 ExcitingGroundStateInput, 
                                 ExcitingPropertiesInput)
from excitingtools.input.input_classes import ExcitingBandStructureInput

We start with defining the lattice of our structure, as well as the positions of the atoms in the unit cell. We also define properties of the atomic species (in our case Gallium and Oxygen). Here, we define in particular the muffin tin radii `rmt`. We also need to provide the `species_path`, which indicates the path to the so called species files. In these files, we define the basis sets. More information on the species files can be found in the [**species file tutorial**](http://exciting.wikidot.com/neon-understanding-species).

In [ ]:
lattice = [[2.882457027, 11.61489328, 0.000000000],
           [-2.882457027, 11.61489328, 0.000000000],
           [0.000000000, 2.600503369, 10.67028355]]

atoms = [{'species': 'Ga', 'position': [0.0000000000, 0.0000000000, 0.0000000000]},
         {'species': 'Ga', 'position': [0.6822620800, 0.6822620800, 0.6280570400]},
         {'species': 'Ga', 'position': [0.2511389000, 0.2511389000, 0.1090277600]},
         {'species': 'Ga', 'position': [0.4311231800, 0.4311231800, 0.5190292800]},
         {'species': 'O', 'position': [0.1679453600, 0.1679453600, 0.8772238900]},
         {'species': 'O', 'position': [0.5143167200, 0.5143167200, 0.7508331500]},
         {'species': 'O', 'position': [0.1760095100, 0.1760095100, 0.4228546600]},
         {'species': 'O', 'position': [0.5062525700, 0.5062525700, 0.2052023800]},
         {'species': 'O', 'position': [0.8449338000, 0.8449338000, 0.5701798300]},
         {'species': 'O', 'position': [0.8373282800, 0.8373282800, 0.0578772100]}]

species_path = "."

structure = ExcitingStructure(atoms, lattice, species_path,
                             autormt=False,
                             species_properties={'Ga': {'rmt': 1.75},
                                                 'O': {'rmt': 1.5}})

In the next step, we define all computational parameters required for a ground-state calculation. 

In [ ]:
groundstate = ExcitingGroundStateInput(do='fromscratch', 
                                       ngridk=[8,8,8], 
                                       xctype='GGA_PBE_SOL', 
                                       gmaxvr=25.0,
                                       lmaxvr=12, 
                                       lmaxapw=12, 
                                       lmaxmat=12, 
                                       rgkmax=8, 
                                       outputlevel='high',
                                       maxscl=50, 
                                       nempty=10)

To understand the attributes that appear in this section of the input file, we will go through all of them in order to clarify their meaning. 
<details> <summary><span style="color:firebrick"><strong>$\Rightarrow$ click here</strong></span></summary>

* <code><span style="color:mediumblue">do</span></code> Decides if the ground state is calculated starting from scratch, using the densities from file, or if its calculation is skipped and only the associated input parameters are read in.
* <code><span style="color:mediumblue">ngridk</span></code> Number of k grid points along the basis vector directions.
* <code><span style="color:mediumblue">xctype</span></code> type of exchange-correlation functional 
* <code><span style="color:mediumblue">gmaxvr</span></code> Maximum length of |G| for expanding the interstitial density and potential.
* <code><span style="color:mediumblue">lmaxvr</span></code> Angular momentum cut-off for the muffin-tin density and potential.
* <code><span style="color:mediumblue">lmaxapw</span></code> Angular momentum cut-off for the APW functions.
* <code><span style="color:mediumblue">lmaxmat</span></code> Angular momentum cut-off for the outer-most loop in the hamiltonian and overlap matrix setup.
* <code><span style="color:mediumblue">rgkmax</span></code> It mplicitly determines the number of basis functions and is one of the crucial parameters for the accuracy of the calculation.
* <code><span style="color:mediumblue">outputlevel</span></code> Specify amount of information which is printed to output files.
* <code><span style="color:mediumblue">maxscl</span></code> Upper limit for the self-consistency loop count.
* <code><span style="color:mediumblue">nempty</span></code> determines the number of empty states used in the construction of the Hamiltonian.

A more detailed descriptions of all the input parameters can be found in the [**input reference**](http://exciting.wikidot.com/ref:input).

Now, that we have the individual parts, we can combine all of them in to one `ExcitingInputXML` object.

In [ ]:
exciting_input = ExcitingInputXML(structure=structure,
                                  groundstate=groundstate,
                                  title="Ga2O3 Ground-state calculation")

This input needs to be written to file. For that we create a new directory

In [ ]:
from pathlib import Path
groundstate_dir = Path("groundstate_tutorial") / "GGA"
groundstate_dir.mkdir(parents=True, exist_ok=True)
input_xml = groundstate_dir / "input.xml"
exciting_input.write(input_xml)

The generated `input.xml` file looks like this.

In [ ]:
print(input_xml.read_text())

## Simulation step

Now, we can run **exciting** using **excitingscripts**. This python library contains a lot of helpful scripts for running exciting and postprocessing exciting outputs. In this tutorial, we use the mock runner to fetch the pre-computed data from [**NOMAD**](https://nomad-lab.eu/prod/v1/gui/search/entries).

In [ ]:
%%bash
cd groundstate_tutorial/GGA
python -m excitingscripts.dpg2026.mock_execute_single groundstate --overwrite
cd ../..

The main output file of exciting is the `INFO.OUT` file. Here, you can find information on the **convergence** of the calculation, including **total energies**, **charge densities**, etc.,. You can now view the `INFO.OUT` in the output directory `groundstate_tutorial/GGA` with your favourite text editor, and have a look at the scf steps as well as valuable information about the quantities important for convergence. Usually, the user can choose between three different levels of the details in the output files, for the case of this tutorial we choose the most detailed variant.

## Post-processing and analysis

Now, we move onto the analysis of our data. In order, to generate the **density of states (DOS)** and **band structure**, we need to add additional elements to our `input.xml` file. For the DOS calculation, we define

In [ ]:
 dos = {'scissor': 0.0,
        'lmirep': True,
        'winddos': [-1.0, 2.0],
        'nwdos': 2000}

For the  band structure calculation, we define the band path and the number of intermediate steps calculated. 

In [ ]:
points = [{'coord': [0.00000000, 0.00000000, 0.00000000], 'label':'GAMMA'},
         {'coord': [0.50000000, 0.50000000, 0.00000000], 'label':'Y'},
         {'coord': [0.60315808, 0.60315808, 0.41000209], 'label':'F'},
         {'coord': [0.50000000, 0.50000000, 0.50000000], 'label':'L'},
         {'coord': [0.74187655, 0.25812345, 0.50000000], 'label':'I', 'breakafter':True},
         {'coord': [0.25812345, -0.25812345, 0.50000000], 'label':'I1'},
         {'coord': [0.00000000, 0.00000000, 0.00000000], 'label':'Z'},
         {'coord': [0.39684192, 0.39684192, 0.58999791], 'label':'F1', 'breakafter':True},
         {'coord': [0.50000000, 0.50000000, 0.00000000], 'label':'Y'},
         {'coord': [0.73364820, 0.26635180, 0.00000000], 'label':'X1', 'breakafter':True},
         {'coord': [0.26635180, -0.26635180, 0.00000000], 'label':'X'},
         {'coord': [0.00000000, 0.00000000, 0.00000000], 'label':'GAMMA'},
         {'coord': [0.50000000, 0.00000000, 0.00000000], 'label':'N', 'breakafter':True},
         {'coord': [0.50000000, 0.00000000, 0.50000000], 'label':'M'},
         {'coord': [0.00000000, 0.00000000, 0.00000000], 'label':'GAMMA'}]

In [ ]:
band_structure = ExcitingBandStructureInput(plot1d={'path':{'steps': 100, 'point': points}})

Since we already computed the ground state, we can also set the `do` attribute to `skip`, in order to avoid a recomputation of it. 

In [ ]:
exciting_input.groundstate.do = 'skip'
exciting_input.properties = ExcitingPropertiesInput(bandstructure=band_structure, dos=dos)

<h3 style="text-align: center; color: firebrick; font-size: 1.2em;">
    At this point, you are now equipped to move onto the subsequent mGGA and hybrid functional tutorials. Therefore, you can choose to continue to compute results for the groundstate or skip to the <a href="#mGGA-section">mGGA</a> / <a href="#hybrid-section">hybrid functional</a> tutorials below
</h3>

With that, we can rewrite the `input.xml` file and run exciting again.

In [ ]:
exciting_input.write(input_xml)

**<span style="color:firebrick">Note</span>**: To avoid overhead, we already fetched the band-structure and DOS ouput files before. So we don't run exciting here. 

The generated DOS and band structure can be plotted using **excitingscripts**. This python library contains a lot of helpful scripts for postprocessing exciting outputs. We start with the DOS.

In [ ]:
%%bash
cd groundstate_tutorial/GGA
python -m excitingscripts.plot.dos
mv PLOT.png DOS.png
cd ../..

In [ ]:
from IPython.display import Image

Image(filename='groundstate_tutorial/GGA/DOS.png') 

In order, to generate a nicer plot, we can explicitly set the energy window to an interesting region using the `-e`, and also choose a window for the DOS with the CLI flag `-db`

In [ ]:
%%bash
cd groundstate_tutorial/GGA
python -m excitingscripts.plot.dos -e -10 12 -db 0 15
mv PLOT.png DOS.png
cd ../..

In [ ]:
Image(filename='groundstate_tutorial/GGA/DOS.png') 

Now, we can go on to visualize the band structure.

In [ ]:
%%bash
cd groundstate_tutorial/GGA
python -m excitingscripts.plot.band_structure
mv PLOT.png BS.png
cd ../..

In [ ]:
Image(filename='groundstate_tutorial/GGA/BS.png') 

Similar to the DOS, we can generate a nicer plot by explicitly setting the energy window to an interesting region using the `-e` option and rescale the plot using the `-s` option. 

In [ ]:
%%bash
cd groundstate_tutorial/GGA
python -m excitingscripts.plot.band_structure -e -9 12 -s 1.5 1
mv PLOT.png BS.png
cd ../..

In [ ]:
Image(filename='groundstate_tutorial/GGA/BS.png') 

<hr style="border:2px solid #DDD"> </hr>

This concludes our tutorial for the GGA groundstate DFT calculation.
Next up, in this notebook we understand how to compute DFT results using meta-GGA functionals. First, let's go through a theoretical overview of this approach, followed by making changes to our already made input file adding parameters for this particular simulation.

<hr style="border:2px solid #DDD"> </hr>

<a id="mGGA-section"></a>
<h1 style="text-align: center; color: firebrick; font-size: 2.5em;">
    Meta-GGA exchange-correlation functionals
</h1>

## Theoretical background
**metaGGA** functionals describe the xc-functional not only in terms of the electron density $n({\bf r})$ and its gradient $\nabla n({\bf r})$ but additionally also in terms of its second derivative $\nabla^2 n({\bf r})$ and/or the **kinetic energy density** (KED)
$$ \tau({\bf r}) = \frac{1}{2} \sum_{n,{\bf k}}^{\rm occ} |\nabla \psi_{n{\bf k}}({\bf r})|^2 \;. $$
Thereby, metaGGA functionals allow for a more accurate description of the derivative discontinuity of xc-energy and thus often provide more accurate predictions of band gaps compared to GGAs with some functionals reaching accuracies comparable to that of hybrid functionals. The inclusion of the KED comes with only a **moderate increase in computational cost** and generally a **slower convergence of the self-consistency cycle**. Still, metaGGAs demand considerably fewer computational resources than hybrids.

## Setting up a metaGGA calculation

In [ ]:
from excitingtools.input.input_classes import ExcitingLibxcInput, ExcitingMggaInput

del(exciting_input.groundstate.xctype)
exciting_input.groundstate.do = 'fromscratch'

exciting_input.groundstate.mgga = ExcitingMggaInput(exchange='XC_MGGA_X_R2SCAN', correlation='XC_MGGA_C_R2SCAN')
exciting_input.groundstate.libxc = ExcitingLibxcInput(exchange='XC_GGA_X_PBE', correlation='XC_GGA_C_PBE')
exciting_input.groundstate.PrelimLinSteps = 8

exciting_input.title = {"title":"Ga2O3 Ground-state calculation (metaGGA)"}

In order to run the metaGGA calculation, we have to make the following changes to the input from the GGA calculation:
1. We unset the `xctype` specified before.
2. We specify the metaGGA functional we want to use by adding the `mgga` element to the groundstate. Here, we use the popular re-regularized SCAN functional. The actual implementation of the different functionals is taken from the *Libxc* library and we specify them by their respective names in *Libxc*.
3. In addition, we have to specify a GGA functional that is only used for core states and for the update of the radial basis functions. Generally, the result is relatively insensitive to the choice of that GGA functional and the PBE functional is good choice for most metaGGA functionals. We do that by setting the `libxc` groundstate element.
4. As metaGGA functionals typically show a worse convergence behavior compared to GGA functionals, we increase to number of preliminary linear mixing steps by setting the attribute `PrelimLinSteps`. This stablelizes the convergence of the electron density by using a linear density mixer for the first few iterations before changing to a more efficient but possibly less stable mixer.

For now we can also remove the information for computing the post-processing `properties`, namely the `bandstructure` and `DOS`. This step is important also for practical reasons as the DFT simulation and post-processing computation requires different parallelization schemes for efficient use of computational resources.


In [ ]:
exciting_input.properties = ExcitingPropertiesInput()

Lets write our modified input to a file have a look at the final input file for the simulation run

In [ ]:
groundstate_dir = Path("groundstate_tutorial") / "metaGGA"
groundstate_dir.mkdir(exist_ok=True)
input_xml = groundstate_dir / "input.xml"
exciting_input.write(input_xml)
print(input_xml.read_text())

## Simulation step

In [ ]:
%%bash
cd groundstate_tutorial/metaGGA
python -m excitingscripts.dpg2026.mock_execute_single metaGGA --overwrite
cd ../..

Similar to the groundstate run, the main output file of the metaGGA run is the `INFO.OUT` file. Here, you can find information on the **convergence** of the calculation, including **total energies**, **charge densities**, etc. You can now view the `INFO.OUT` in the output directory `groundstate_tutorial/metaGGA` with your favourite text editor, and have a look at the scf steps as well as valuable information about the quantities important for convergence. Usually, the user can choose between three different levels of the details in the output files, for the case of this tutorial we choose the most detailed variant.

## Post-processing and analysis

Next up, let us redefine the post-processing elements in our input file, and set the `do` parameter such that it skips the simulation run and we use the same band structure and dos definitions as in the GGA calculation before.

In [ ]:
exciting_input.groundstate.do = 'skip'
exciting_input.properties = ExcitingPropertiesInput(bandstructure=band_structure, dos=dos)

With that, we can rewrite the `input.xml` file.

In [ ]:
exciting_input.write(input_xml)

**<span style="color:firebrick">Note</span>**: To avoid overhead, we already fetched the band-structure and DOS ouput files before. So we don't run exciting here. 

Let us now visualize the band structure computed with meta-GGAs using **excitingscripts**. 

In [ ]:
%%bash
cd groundstate_tutorial/metaGGA
python -m excitingscripts.plot.band_structure -e -9 12 -s 1.5 1
mv PLOT.png BS.png
cd ../..

In [ ]:
Image(filename='groundstate_tutorial/metaGGA/BS.png') 


By giving the explicit path to the band structure ouptut files using the `-d` option, we are also able to compare the band structures computed with GGA and meta-GGA functionals in one plot. 

In [ ]:
%%bash
cd groundstate_tutorial
python -m excitingscripts.plot.band_structure -e -9 12 -s 1.5 1 -d ./GGA ./metaGGA -ll "GGA" "mGGA" -t "Ga2O3 bandstructure comparison with GGA & metaGGA functionals"
mv PLOT.png BS_compare_mGGA_GGA.png
cd ..

In [ ]:
Image(filename='groundstate_tutorial/BS_compare_mGGA_GGA.png') 


<hr style="border:2px solid #DDD"> </hr>

This concludes our tutorial for the metaGGA groundstate DFT calculation.
Next up, in this notebook we understand how to compute DFT results using hybrid functionals. First, let's go through a theoretical overview of this approach, followed by making changes to our already made input file adding parameters for this particular simulation.

<hr style="border:2px solid #DDD"> </hr>

<a id="hybrid-section"></a>
<h1 style="text-align: center; color: firebrick; font-size: 2.5em;">
    Hybrid exchange-correlation functionals
</h1>

## Theoretical background

The hybrid exchange-correlation functionals are constructed by substituting a fraction of local/semi-local exchange by a fraction of non-local exact-exchange:

\begin{equation}
E^{hyb}_{xc} = E^L_{xc}+\alpha \ (E^{NL}_x-E^L_x).
\end{equation}

The non-local exact exchange is often computed employing the Hartree-Fock (**HF**) method. The type of the **DFT** local/semi-local functional and the fraction of non-local exact exchange substituted ($\alpha$) vary depending on the hybrid functional type.

The hybrid functionals **PBE0** and **HSE** are implemented in **`exciting`**. In today's session, we focus on the hybrid HSE functional, which is a screened hybrid functional of the following form: 

\begin{equation}
E^{HSE}_{xc} = E^{PBE}_{xc}+\alpha \ [E^{HF,SR}_x(\omega)-E^{PBE,SR}_x(\omega)].
\end{equation}

In **HSE**, **PBE** is used as semi-local functional, but only the short-range part of the non-local exact exchange is substituted. In fact, in **HSE** the amount of non-local exact exchange is determined by both the mixing parameter $\alpha$ = **<span style="color:firebrick">0.25</span>** and the screening screening parameter $\omega$, which defines the cutoff between the short-range and the long-range part of the Coulomb operator through the error function and its complementary:

\begin{equation}
v(r)=v^{SR}(r,\omega)+v^{LR}(r,\omega)=\frac{\text{erfc} \ (\omega r)}{r}+\frac{\text{erf} \ (\omega r)}{r}.
\end{equation}

Natively, the version of **HSE** which is implemented in **`exciting`** is **HSE06**, in which the **PBE** and **HF** contributions have the same value of $\omega$ = **<span style="color:firebrick">0.11 $a_0^{-1}$</span>**.

## Setting up a hybrid functional calculation

In [ ]:
del(exciting_input.groundstate.mgga)
del(exciting_input.groundstate.libxc)
exciting_input.groundstate.do = "fromscratch"
exciting_input.groundstate.xctype = "HYB_HSE"
exciting_input.groundstate.nempty = 400
exciting_input.groundstate.Hybrid = {'exchangetype':'HF', 'excoeff':0.25, 'omega':0.11}
exciting_input.title = {"title":"Ga2O3 Ground-state calculation (Hybrids)"}

Below, you can find a detailed explanation of the additional input parameters needed:

1. We set the `do` parameter to perform the calculation from scratch.
2. We set the `xctype` to the hybrid functional of choice (`HYB_PBE0` or `HYB_HSE`)
3. `nempty` governs the number of additional empty eigenstates beyond those required for charge neutrality. Usually, we require a high `nempty` for achieving convergence with hybrid functionals.
4. `excoeff`corresponds to the $\alpha$ parameter, which is taken as a standard value of 25% mixing of the PBE exchange energy with the HF energy defined by: `exchangetype`

For now we can also remove the information for computing the post-processing `properties`, namely the `bandstructure` and `DOS`. This step is important also for practical reasons as the DFT simulation and post-processing computation requires different parallelization schemes for efficient use of computational resources.

In [ ]:
exciting_input.properties = ExcitingPropertiesInput()

Lets write our modified input to a file and have a look at the final input file for the simulation run

In [ ]:
groundstate_dir = Path("groundstate_tutorial") / "hybrids"
groundstate_dir.mkdir(exist_ok=True)
input_xml = groundstate_dir / "input.xml"
exciting_input.write(input_xml)
print(input_xml.read_text())

## Simulation step

In [ ]:
%%bash
cd groundstate_tutorial/hybrids
python -m excitingscripts.dpg2026.mock_execute_single hybrid --overwrite
cd ../..

We can now checkout the output of our calculation with the `INFO.OUT` file in the `groundstate_tutorial/hybrids` directory and subsequently move onto the postprocessing routines like computing the bandstructure and dos. However, due to the high complexity of the hybrid functional approximation, the calculation of electronic properties on a fine grid in reciprocal space is very expensive and is prohibited. 

To circumvent this issue, we compute **Maximally Localized Wannier Functions (MLWFs)** to interpolate the reciprocal space. This is an advanced scheme and we recommend checking out the detailed tutorial available at the [exciting webpage](https://www.exciting-code.org/) at your leisure. For our case, let us now setup computation of the wannier functions first.

## Post-processing

In [ ]:
exciting_input.groundstate.do = 'skip'

groups = [{'method' : 'opfmax', 'fst' : 43, 'lst' : 60, 'nproj' : 100},
          {'method' : 'disFull', 'innerwindow' : [0.0, 0.7349864496307141], 'outerwindow' : [0.0, 1.102479674446071], 'nproj' : 200}
          ]

wannier = {'input': 'hybrid',
          'do' : 'fromscratch',
          'group' : groups 
          }

exciting_input.properties = ExcitingPropertiesInput(wannier=wannier)

Let us briefly understand each of these parameters:

1. We declare the class of exchange-correlation functional used, indicated here by the `input: 'hybrid'`.
1. We set the `do` parameter to `skip` to indicate that the DFT calculation is already done and we can utilize the results from the simulation.
3. To define the `wannier` attribute, we must declare 2 groups of bands:
- The first group describes an isolated set of 18 bands, indicating the last bunch of valence bands. The `method` describes the method we use of the initial guess of the wannier functions, which in this case is the so called optimized projection functions (OPF).
- The second group describes the conduction bands in the two energy windows, and triggers a four step disentanglement process, triggered by the `method`: disFull

**<span style="color:firebrick">Note</span>**: You can read more about this implementation and the computation schemes for wannier functions [here](https://journals.aps.org/prb/abstract/10.1103/PhysRevB.101.235102)

Let us now save our changes to the input file and see how it looks.  

In [ ]:
exciting_input.write(input_xml)
print(input_xml.read_text())

**<span style="color:firebrick">Note</span>**: To avoid overhead, we already fetched the computed wannier functions already. So we don't run exciting here.

Information about the computed wannier functions is stored in the file `WANNIER_INFO.OUT`, but since we focus on computing the bandstructure and dos properties within the scope of this session. Let us now have a look at these properties.

let us redefine the post-processing elements in our input file, and set the `do` parameter such that it skips the simulation run

## Analysis

In [ ]:
exciting_input.properties.wannier.do = 'fromfile'
band_structure.wannier = True
dos = {'wannier': True,
       'ngridkint': [60, 60, 60],
       'lmirep': True,
       'ngrdos': 500,
       'lmirep': True,
       'inttype': 'tetra',
       'winddos': [-1.0, 1.0],
       'nwdos': 1000}
wanniergap = {'ngridkint': [60, 60, 60]}
exciting_input.properties = ExcitingPropertiesInput(wannier=wannier, bandstructure=band_structure, wanniergap=wanniergap, dos=dos)

In [ ]:
exciting_input.write(input_xml)
print(input_xml.read_text())

With that, we can rewrite the `input.xml` file and run exciting again. 

**<span style="color:firebrick">Note</span>**: To avoid overhead, we already fetched the band-structure and DOS ouput files before. So we don't run exciting here. 

The generated DOS and band structure can be plotted again using **excitingscripts**. 

In [ ]:
%%bash
cd groundstate_tutorial/hybrids
python -m excitingscripts.plot.dos -e -15 12 -a WA
mv PLOT.png DOS.png
cd ../..

In [ ]:
Image(filename='groundstate_tutorial/hybrids/DOS.png') 

By giving the explicit path to the band structure ouptut files using the `-d` option, we are also able to compare the band structures computed with GGA, metaGGA and hybrid functionals in one plot.

In [ ]:
%%bash
cd groundstate_tutorial/hybrids
python -m excitingscripts.plot.band_structure -e -9 12 -s 1.5 1 -a WA
mv PLOT.png BS.png
cd ../..

In [ ]:
Image(filename='groundstate_tutorial/hybrids/BS.png') 

By giving the explicit path to the band structure ouptut files using the `-d` option, we are also able to compare the band structures computed with GGA, meta-GGA, and hybrid functionals in one plot.  

In [ ]:
%%bash
cd groundstate_tutorial
python -m excitingscripts.plot.band_structure -a KS KS WA -e -9 12 -s 1.5 1 -d ./GGA ./metaGGA ./hybrids -ll "GGA" "metaGGA" "hybrids"
mv PLOT.png BS_compare_hybrids_mGGa_GGA.png
cd ..

In [ ]:
Image(filename='groundstate_tutorial/BS_compare_hybrids_mGGa_GGA.png') 

Lets zoom in a bit to visualize the band dispersion even better

In [ ]:
%%bash
cd groundstate_tutorial
python -m excitingscripts.plot.band_structure -a KS KS WA -e -4 7 -s 1.5 1 -d ./GGA ./metaGGA ./hybrids -ll "GGA" "metaGGA" "hybrids"
mv PLOT.png BS_compare_hybrids_mGGa_GGA.png
cd ..

In [ ]:
Image(filename='groundstate_tutorial/BS_compare_hybrids_mGGa_GGA.png') 

<hr style="border:2px solid #DDD"> </hr>
This concludes our tutorial for the groundstate DFT calculation with different xc-functionals, you are encouraged to checkout beyond DFT methods via other tutorials in this suite.
<hr style="border:2px solid #DDD"> </hr>